In [ ]:
import os
import numpy as np
from IPython.display import display, HTML, Image
from PIL import Image as PILImage, ImageDraw, ImageFont
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import re
import warnings
import geopandas as gpd
from shapely.geometry import shape, Polygon, MultiPolygon, box, Point
import pandas as pd
from rasterio import features
from rasterio.transform import Affine
import json
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import seaborn as sns
import pickle
import sys
import matplotlib.patches as mpatches
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import PCA
import joblib

# Suppress font warnings / Потискане на предупреждения за шрифтове
warnings.filterwarnings('ignore', category=UserWarning)

# ==================== CONFIGURATION / КОНФИГУРАЦИЯ ====================
# (English) Path to the CSV with fire metadata (originally fires_suggestion.csv)
# (Bulgarian) Път до CSV файла с метаданни за пожарите (първоначално fires_suggestion.csv)
file_path = r'D:\data\master_thesis\input\fires_suggestion.csv'

# (EN) Directory with Sentinel‑2 composite images
# (BG) Директория с композитни изображения Sentinel‑2
PREVIEW_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures'

# (EN) Output directory for SVC classification
# (BG) Изходна директория за класификация с SVC
CLASSIFIED_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_report\svc'

# (EN) Vector exports sub‑directory
# (BG) Поддиректория за векторен експорт
VECTOR_EXPORT_DIR = os.path.join(CLASSIFIED_DIR, 'vector_exports')

# (EN) Directory with training polygons (GeoPackage)
# (BG) Директория с тренировъчни полигони (GeoPackage)
POLYGON_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\polygons'

# (EN) Directory with urban mask GeoPackages
# (BG) Директория с урбанизирани маски (GeoPackage)
URBAN_MASK_DIR = r'D:\data\master_thesis\exports\sentinel2_fire_images\classification_pictures\geopackages'

os.makedirs(CLASSIFIED_DIR, exist_ok=True)
os.makedirs(VECTOR_EXPORT_DIR, exist_ok=True)

# (EN) Class definitions – id, name, RGB colour, polygon file prefix
# (BG) Дефиниции на класовете – идентификатор, име, RGB цвят, префикс на полигоновия файл
CLASSES = {
    1: {'name': 'Urban',              'color': (255,   0,   0), 'polygon_file': 'urban'},
    2: {'name': 'Water',              'color': (  0,   0, 255), 'polygon_file': 'water'},
    3: {'name': 'Bare and Urban Territories', 'color': (139,  69,  19), 'polygon_file': 'bare_lands'},
    4: {'name': 'Field/Agriculture',  'color': (255, 255,   0), 'polygon_file': 'field_agriculture'},
    5: {'name': 'Coniferous Forest',  'color': (  0, 100,   0), 'polygon_file': 'coniferous'},
    6: {'name': 'Deciduous Forest',   'color': (  0, 255,   0), 'polygon_file': 'forest_deciduous'}
}

# ==================== CRS HANDLING / ОБРАБОТКА НА КООРДИНАТНА СИСТЕМА ====================
_crs_cache = None

def _load_crs_csv():
    """
    (EN) Load the fires_suggestion.csv once and cache it.
    (BG) Зарежда fires_suggestion.csv еднократно и го кешира.
    """
    global _crs_cache
    if _crs_cache is None:
        try:
            _crs_cache = pd.read_csv(file_path)
            print(f"✅ Loaded CRS information from {file_path} ({len(_crs_cache)} rows)")
        except Exception as e:
            print(f"❌ Could not load CRS file: {e}")
            _crs_cache = pd.DataFrame()
    return _crs_cache

def get_crs_for_fire(fire_num):
    """
    (EN) Retrieve the target CRS from the CSV for a given fire number.
    (BG) Извлича целевия CRS от CSV за даден номер на пожар.
    """
    df = _load_crs_csv()
    if df.empty:
        return None

    cols_lower = {col.lower(): col for col in df.columns}
    fire_col = None
    for candidate in ['fire_number', 'fire_num', 'fire_id', 'firenumber']:
        if candidate in cols_lower:
            fire_col = cols_lower[candidate]
            break
    if fire_col is None:
        for col in df.columns:
            low = col.lower()
            if 'fire' in low and ('id' in low or 'number' in low):
                fire_col = col
                break
    if fire_col is None:
        print(f"⚠ Warning: Could not find a fire identifier column. Available: {df.columns.tolist()}")
        return None

    matches = df[df[fire_col] == fire_num]
    if matches.empty:
        print(f"⚠ Warning: Fire number {fire_num} not found in CSV")
        return None

    crs_val = matches['crs'].dropna().iloc[0] if not matches['crs'].dropna().empty else None
    if crs_val is None:
        print(f"⚠ Warning: No CRS value for fire {fire_num}")
        return None
    print(f"  📌 CRS from CSV for fire {fire_num}: {crs_val}")
    return str(crs_val)

def reproject_classification(classification, src_transform, src_crs, dst_crs):
    """
    (EN) Reproject classification raster to another CRS.
    (BG) Препроектира класификационен растер към друг CRS.
    """
    src_height, src_width = classification.shape
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, dst_crs, src_width, src_height,
        left=src_transform[2],
        bottom=src_transform[5] + src_height * src_transform[4],
        right=src_transform[2] + src_width * src_transform[0],
        top=src_transform[5]
    )
    dst_arr = np.zeros((dst_height, dst_width), dtype=classification.dtype)
    reproject(source=classification, destination=dst_arr,
              src_transform=src_transform, src_crs=src_crs,
              dst_transform=dst_transform, dst_crs=dst_crs,
              resampling=Resampling.nearest)
    return dst_arr, dst_transform

def safe_transform_points(points, src_crs, dst_crs):
    """
    (EN) Transform coordinates between CRS, return None on failure.
    (BG) Трансформира координати между CRS, връща None при грешка.
    """
    try:
        from rasterio.warp import transform
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        new_xs, new_ys = transform(src_crs, dst_crs, xs, ys)
        return [(x, y) for x, y in zip(new_xs, new_ys)]
    except Exception as e:
        print(f"    ⚠ CRS transform failed ({src_crs}→{dst_crs}): {e}")
        return None

def _fix_polygon_crs_if_needed(gdf, file_name):
    """
    (EN) If CRS is projected but coordinates look geographic, override to EPSG:4326.
    (BG) Ако CRS е проектиран, но координатите изглеждат географски, задава EPSG:4326.
    """
    if gdf.crs is None or gdf.crs.is_projected:
        try:
            minx, miny, maxx, maxy = gdf.total_bounds
            if -180 <= minx <= 180 and -90 <= miny <= 90 and -180 <= maxx <= 180 and -90 <= maxy <= 90:
                print(f"    ⚠ {file_name}: CRS is {gdf.crs} but coords look like degrees → forcing EPSG:4326")
                gdf = gdf.set_crs('EPSG:4326', allow_override=True)
        except Exception:
            pass
    return gdf

# ==================== HELPER FUNCTIONS / ПОМОЩНИ ФУНКЦИИ ====================
def extract_fire_number_from_filename(filename):
    """
    (EN) Extract fire number from filename.
    (BG) Извлича номер на пожар от име на файл.
    """
    m = re.search(r'fire[_\s]?(\d+)', filename.lower())
    return int(m.group(1)) if m else None

def get_fire_id_formats(fire_num):
    """
    (EN) Return different string representations of the fire ID.
    (BG) Връща различни текстови представяния на идентификатора на пожара.
    """
    return {
        'fire_num': str(fire_num),
        'fire_01': f"fire_{fire_num:02d}",
        'fire_1': f"fire_{fire_num}",
        'fire1': f"fire{fire_num}"
    }

def find_polygon_directory(fire_id_dict):
    """
    (EN) Locate the polygon directory for a given fire.
    (BG) Намира директорията с полигони за даден пожар.
    """
    possible = [
        os.path.join(POLYGON_DIR, fire_id_dict['fire_01']),
        os.path.join(POLYGON_DIR, fire_id_dict['fire_1']),
        os.path.join(POLYGON_DIR, fire_id_dict['fire1'])
    ]
    for d in possible:
        if os.path.exists(d):
            return d
    print(f"  - Checking for loose polygon files...")
    return POLYGON_DIR

def load_polygon_samples(fire_id_dict):
    """
    (EN) Load training polygons, sample random points inside, return (points, labels, CRS).
    (BG) Зарежда тренировъчни полигони, взима случайни точки и връща (точки, етикети, CRS).
    """
    fire_num = fire_id_dict['fire_num']
    poly_dir = find_polygon_directory(fire_id_dict)
    polygon_crs = None

    def _process_directory(base_dir, file_patterns):
        nonlocal polygon_crs
        pts, lbls = [], []
        for class_id, info in CLASSES.items():
            found = False
            for pat in file_patterns:
                try:
                    fname = pat % (fire_num, info['polygon_file'])
                except TypeError:
                    fname = pat.format(fire_num=fire_num, poly=info['polygon_file'])
                fpath = os.path.join(base_dir, fname)
                if os.path.exists(fpath):
                    try:
                        gdf = gpd.read_file(fpath)
                        gdf = _fix_polygon_crs_if_needed(gdf, fname)
                        if len(gdf) == 0:
                            print(f"    ⚠ No features in {fname}")
                            continue
                        if polygon_crs is None:
                            polygon_crs = gdf.crs
                        print(f"  - Loading {fname}...")
                        for geom in gdf.geometry:
                            if geom.geom_type == 'Polygon':
                                minx, miny, maxx, maxy = geom.bounds
                                n = min(100, max(10, int(geom.area / 10000)))
                                for _ in range(n):
                                    for _ in range(100):
                                        pt = Point(np.random.uniform(minx, maxx),
                                                   np.random.uniform(miny, maxy))
                                        if geom.contains(pt):
                                            pts.append((pt.x, pt.y))
                                            lbls.append(int(class_id))
                                            break
                        print(f"    ✓ Loaded {len(gdf)} {info['name']} polygons ({n} points each)")
                        found = True
                        break
                    except Exception as e:
                        print(f"    ❌ Error loading {fname}: {e}")
            if not found:
                print(f"  - {info['name']} polygons not found (skipping)")
        return pts, lbls

    if poly_dir == POLYGON_DIR:
        # Loose files / разпръснати файлове
        patterns = [
            f"fire%s_%s.gpkg",           # e.g., fire14_water.gpkg
            f"fire_%s_%s.gpkg",          # fire_14_water.gpkg
            f"fire_%02d_%s.gpkg"         # fire_014_water.gpkg (zero padded)
        ]
    else:
        patterns = [
            f"fire%s_%s.gpkg",
            f"fire_%s_%s.gpkg",
            f"%s_%s.gpkg",
            f"%s.gpkg"
        ]
    pts, lbls = _process_directory(poly_dir, patterns)

    if not pts:
        print(f"  ❌ No polygon files found for fire {fire_num}")
        return None, None, None
    if polygon_crs is None:
        print("  ⚠ No CRS defined; assuming EPSG:4326")
        polygon_crs = 'EPSG:4326'
    return np.array(pts), np.array(lbls), polygon_crs

def load_urban_mask(fire_id_dict):
    """
    (EN) Load urban mask GeoPackage for a fire.
    (BG) Зарежда маска на урбанизирани територии за пожар.
    """
    fire_num = fire_id_dict['fire_num']
    patterns = [
        f"fire{fire_num}_urban_masks.gpkg",
        f"fire_{fire_num}_urban_masks.gpkg",
        f"fire_{int(fire_num):02d}_urban_masks.gpkg",
        f"fire{fire_num}_urban_mask.gpkg",
        f"fire_{fire_num}_urban_mask.gpkg",
        f"urban_mask_{fire_num}.gpkg",
        f"urban_masks_fire{fire_num}.gpkg",
        f"urban_masks.gpkg"
    ]
    print(f"  - Looking for urban masks in {URBAN_MASK_DIR}")
    if os.path.exists(URBAN_MASK_DIR):
        avail = [f for f in os.listdir(URBAN_MASK_DIR) if 'urban' in f.lower() and f.endswith('.gpkg')]
        print(f"  - Available: {avail}" if avail else "  - No urban mask files found")
    else:
        print("  - Urban mask directory not found")

    for fname in patterns:
        path = os.path.join(URBAN_MASK_DIR, fname)
        if os.path.exists(path):
            try:
                gdf = gpd.read_file(path)
                if len(gdf) > 0:
                    print(f"  ✓ Urban mask loaded: {fname} ({len(gdf)} polygons)")
                    if gdf.crs is None:
                        print("  ⚠ No CRS; assuming EPSG:4326")
                        gdf = gdf.set_crs('EPSG:4326', allow_override=True)
                    return gdf
                else:
                    print(f"  ⚠ Empty mask: {fname}")
            except Exception as e:
                print(f"  ❌ Error loading {fname}: {e}")
    print(f"  ⚠ No urban mask found for fire {fire_num}")
    return None

def extract_band_values_at_points(raster_path, points):
    """
    (EN) Extract all band values at geographic points.
    (BG) Извлича стойностите на всички канали в зададени географски точки.
    """
    with rasterio.open(raster_path) as src:
        vals, idx = [], []
        print(f"    Extracting values for {len(points)} points...")
        for i, (x, y) in enumerate(points):
            try:
                row, col = src.index(x, y)
                if 0 <= row < src.height and 0 <= col < src.width:
                    pix = [src.read(b+1)[row, col] for b in range(src.count)]
                    vals.append(pix)
                    idx.append(i)
            except Exception:
                pass
        print(f"    ✓ Valid points: {len(idx)}/{len(points)}")
        return np.array(vals), np.array(idx)

def identify_sentinel2_bands_correctly(bands, src_descriptions=None):
    """
    (EN) Detect Blue, Green, Red, NIR bands either from descriptions or heuristically.
    (BG) Открива синия, зеления, червения и NIR канали по описания или евристично.
    """
    print(f"  - Number of bands: {len(bands)}")
    if src_descriptions:
        print(f"  - Band descriptions: {src_descriptions}")
    if src_descriptions:
        band_map = {}
        for i, desc in enumerate(src_descriptions):
            if not desc:
                continue
            desc = desc.upper()
            if 'B02' in desc or desc == 'B2':
                band_map['blue'] = bands[i]
                print(f"    Found Blue (B02) at band {i+1}")
            elif 'B03' in desc or desc == 'B3':
                band_map['green'] = bands[i]
                print(f"    Found Green (B03) at band {i+1}")
            elif 'B04' in desc or desc == 'B4':
                band_map['red'] = bands[i]
                print(f"    Found Red (B04) at band {i+1}")
            elif 'B08' in desc or desc == 'B8':
                band_map['nir'] = bands[i]
                print(f"    Found NIR (B08) at band {i+1}")
            elif 'B8A' in desc and 'nir' not in band_map:
                band_map['nir'] = bands[i]
                print(f"    Found NIR (B8A) at band {i+1}")
        if len(band_map) == 4:
            print("  - Successfully identified bands from descriptions")
            return band_map

    if len(bands) >= 15:
        print("  - Detected Sentinel-2 L2A product (15 bands)")
        return {'blue': bands[2], 'green': bands[3], 'red': bands[4], 'nir': bands[8]}
    if len(bands) >= 12:
        print("  - Identified as 12-band Sentinel-2")
        return {'blue': bands[1], 'green': bands[2], 'red': bands[3], 'nir': bands[7]}
    elif len(bands) >= 4:
        band_means = [np.mean(b) for b in bands[:4]]
        nir_idx = np.argmax(band_means)
        red_idx = np.argmin(band_means[:3])
        remaining = [i for i in range(4) if i not in [nir_idx, red_idx]]
        return {'blue': bands[remaining[0]], 'green': bands[remaining[1]], 'red': bands[red_idx], 'nir': bands[nir_idx]}
    else:
        return {'blue': bands[0] if len(bands) > 0 else None,
                'green': bands[1] if len(bands) > 1 else None,
                'red': bands[2] if len(bands) > 2 else None,
                'nir': bands[3] if len(bands) > 3 else None}

def calculate_indices(band_info):
    """
    (EN) Calculate NDVI and NDWI from band arrays.
    (BG) Изчислява NDVI и NDWI от масивите на каналите.
    """
    indices = {}
    if 'red' in band_info and 'nir' in band_info:
        red = band_info['red'].astype(np.float32)
        nir = band_info['nir'].astype(np.float32)
        denom = nir + red
        valid = denom > 0
        ndvi = np.zeros_like(red, dtype=np.float32)
        ndvi[valid] = (nir[valid] - red[valid]) / denom[valid]
        indices['ndvi'] = np.clip(ndvi, -1, 1)
    if 'green' in band_info and 'nir' in band_info:
        green = band_info['green'].astype(np.float32)
        nir = band_info['nir'].astype(np.float32)
        denom = green + nir
        valid = denom > 0
        ndwi = np.zeros_like(green, dtype=np.float32)
        ndwi[valid] = (green[valid] - nir[valid]) / denom[valid]
        indices['ndwi'] = np.clip(ndwi, -1, 1)
    return indices

def create_true_color_rgb(band_info):
    """
    (EN) Create true‑colour RGB image with 2–98% stretch.
    (BG) Създава истинско цветно RGB изображение с 2–98% разтягане.
    """
    if 'red' in band_info and 'green' in band_info and 'blue' in band_info:
        def enhance(band):
            p2, p98 = np.percentile(band, 2), np.percentile(band, 98)
            return ((np.clip((band - p2) / (p98 - p2), 0, 1)) * 255).astype(np.uint8)
        return np.dstack((enhance(band_info['red']),
                          enhance(band_info['green']),
                          enhance(band_info['blue'])))
    else:
        print("❌ Missing bands for true color RGB")
        return None

# ==================== CLASSIFICATION FUNCTIONS / ФУНКЦИИ ЗА КЛАСИФИКАЦИЯ ====================
def train_svc_classifier(raster_path, training_points, training_labels, polygon_crs=None):
    """
    (EN) Train SVC classifier using spectral bands and indices.
         Attempts multiple CRS transformations to find valid training pixels.
    (BG) Обучава SVC класификатор с помощта на спектрални канали и индекси.
         Пробва различни CRS трансформации за намиране на валидни тренировъчни пиксели.
    """
    print("  - Extracting band values at training points...")
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        raster_bounds = src.bounds
        print(f"    Raster CRS: {raster_crs}, Bounds: {raster_bounds}")

    best_valid = 0
    best_points = training_points

    # Strategy 0: if CRS match, use directly
    if polygon_crs is None or str(raster_crs) == str(polygon_crs):
        _, valid0 = extract_band_values_at_points(raster_path, training_points)
        if len(valid0) > best_valid:
            best_valid = len(valid0)
            best_points = training_points

    # Strategy 1: transform if CRS different
    if polygon_crs is not None and str(raster_crs) != str(polygon_crs):
        transformed = safe_transform_points(training_points, polygon_crs, raster_crs)
        if transformed is not None:
            _, valid1 = extract_band_values_at_points(raster_path, transformed)
            if len(valid1) > best_valid:
                best_valid = len(valid1)
                best_points = transformed

    # Strategy 2: assume original coordinates already in raster CRS (ignore polygon CRS)
    if best_valid == 0:
        print("    ⚠ Trying direct use of original coordinates (ignoring polygon CRS)...")
        _, valid2 = extract_band_values_at_points(raster_path, training_points)
        if len(valid2) > best_valid:
            best_valid = len(valid2)
            best_points = training_points

    # Strategy 3: if polygon CRS is projected, try interpreting as EPSG:4326
    if best_valid == 0 and polygon_crs is not None:
        poly_crs_obj = rasterio.crs.CRS.from_user_input(polygon_crs)
        if poly_crs_obj.is_projected:
            print("    ⚠ Trying interpretation as EPSG:4326 (geographic) and transform...")
            transformed3 = safe_transform_points(training_points, 'EPSG:4326', raster_crs)
            if transformed3 is not None:
                _, valid3 = extract_band_values_at_points(raster_path, transformed3)
                if len(valid3) > best_valid:
                    best_valid = len(valid3)
                    best_points = transformed3

    if best_valid == 0:
        print("  ❌ No valid training points after all strategies. Skipping classifier training.")
        return None, None, None

    # Extract final values with best points
    X, valid_indices = extract_band_values_at_points(raster_path, best_points)
    if len(X) == 0:
        print("  ❌ No valid training points found.")
        return None, None, None

    y = training_labels[valid_indices]
    # Clean labels / Почистване на етикети
    y_clean, valid_lbl_idx = [], []
    for i, lbl in enumerate(y):
        try:
            lbl_int = int(float(lbl))
            if lbl_int in CLASSES:
                y_clean.append(lbl_int)
                valid_lbl_idx.append(i)
            else:
                print(f"    ⚠ Skipping invalid class ID: {lbl}")
        except Exception:
            print(f"    ⚠ Skipping non‑numeric label: {lbl}")
    if len(y_clean) == 0:
        print("  ❌ No valid class labels")
        return None, None, None
    X_clean = X[valid_lbl_idx]
    y_clean = np.array(y_clean)

    uniq, cnts = np.unique(y_clean, return_counts=True)
    print("  - Class distribution:")
    for cls_id, cnt in zip(uniq, cnts):
        print(f"    {CLASSES[cls_id]['name']}: {cnt} samples")
    min_samples = 2
    ok_cls = [c for c, n in zip(uniq, cnts) if n >= min_samples]
    if len(ok_cls) < 2:
        print(f"  ❌ Need at least 2 classes with {min_samples}+ samples")
        return None, None, None
    mask = np.isin(y_clean, ok_cls)
    X_filt = X_clean[mask]
    y_filt = y_clean[mask]

    # Add spectral indices / Добавяне на спектрални индекси
    with rasterio.open(raster_path) as src:
        bands = [src.read(i+1) for i in range(src.count)]
        band_info = identify_sentinel2_bands_correctly(bands, src.descriptions if src.descriptions else None)
        indices = calculate_indices(band_info)

    if 'ndvi' in indices or 'ndwi' in indices:
        print("  - Adding spectral indices...")
        X_with_idx = []
        filtered_pts_idx = valid_indices[valid_lbl_idx][mask]
        filtered_pts = best_points[filtered_pts_idx]
        with rasterio.open(raster_path) as src:
            for i, (x, y) in enumerate(filtered_pts):
                row, col = src.index(x, y)
                if 0 <= row < src.height and 0 <= col < src.width:
                    feat = X_filt[i].tolist()
                    if 'ndvi' in indices:
                        feat.append(indices['ndvi'][row, col])
                    if 'ndwi' in indices:
                        feat.append(indices['ndwi'][row, col])
                    X_with_idx.append(feat)
        X_filt = np.array(X_with_idx)

    # Standardize features (important for SVC)
    print("  - Standardizing features...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filt)

    # Optional PCA dimensionality reduction
    pca = None
    if X_scaled.shape[1] > 10:
        print(f"  - Reducing dimensionality from {X_scaled.shape[1]} to 10 features...")
        pca = PCA(n_components=min(10, X_scaled.shape[1]), random_state=42)
        X_scaled = pca.fit_transform(X_scaled)
        print(f"    ✓ Explained variance: {np.sum(pca.explained_variance_ratio_):.3f}")

    # Train/test split / Разделяне на тренировъчни и тестови данни
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_filt, test_size=0.2,
                                                   random_state=42, stratify=y_filt)
    except ValueError as e:
        print(f"  ⚠ Stratification failed ({e}), using random split")
        X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_filt, test_size=0.2, random_state=42)

    # Train SVC classifier / Обучение на SVC класификатора
    svc = OneVsRestClassifier(
        SVC(C=1.0, kernel='rbf', gamma='scale', probability=True,
            random_state=42, cache_size=500),
        n_jobs=-1
    )
    svc.fit(X_tr, y_tr)
    y_pred = svc.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    print(f"\n  - SVC accuracy: {acc:.3f}")
    target_names = [CLASSES[c]['name'] for c in sorted(np.unique(y_filt))]
    print("\n  - Classification Report:")
    print(classification_report(y_te, y_pred, target_names=target_names))
    print("\n  - Confusion Matrix:\n", confusion_matrix(y_te, y_pred))
    return svc, scaler, pca

def classify_raster_with_svc(raster_path, classifier, scaler, pca, urban_mask=None):
    """
    (EN) Apply SVC to entire raster, optionally forcing urban mask to class 3.
    (BG) Прилага SVC върху целия растер, опционално задава урбанизираните зони към клас 3.
    """
    print("  - Classifying raster...")
    with rasterio.open(raster_path) as src:
        bands = [src.read(i+1) for i in range(src.count)]
        height, width = bands[0].shape
        print(f"  - Raster dimensions: {height} x {width}")
        band_info = identify_sentinel2_bands_correctly(bands, src.descriptions if src.descriptions else None)
        indices = calculate_indices(band_info)

        # Build feature matrix
        band_flat = [b.reshape(-1, 1) for b in bands]
        X = np.hstack(band_flat)
        if 'ndvi' in indices:
            X = np.hstack([X, indices['ndvi'].reshape(-1, 1)])
        if 'ndwi' in indices:
            X = np.hstack([X, indices['ndwi'].reshape(-1, 1)])

        # Scale features
        print("  - Scaling features...")
        X_scaled = scaler.transform(X)

        # PCA if used
        if pca is not None:
            print("  - Applying PCA...")
            X_scaled = pca.transform(X_scaled)

        # Predict in batches (SVC is slower)
        batch_size = 50000
        y_pred = np.zeros(height * width, dtype=np.uint8)
        for i in range(0, len(X_scaled), batch_size):
            end = min(i + batch_size, len(X_scaled))
            y_pred[i:end] = classifier.predict(X_scaled[i:end])
            print(f"    Progress: {end/len(X_scaled)*100:.1f}%", end='\r')
        print("    Progress: 100.0%")
        classification = y_pred.reshape(height, width)

        # Urban mask application / Прилагане на урбанизирана маска
        bare_before = np.sum(classification == 3)
        print(f"    Bare and Urban BEFORE urban mask: {bare_before:,} pixels")
        if urban_mask is not None:
            print("  - Applying urban mask...")
            if urban_mask.crs != src.crs:
                print(f"    Transforming urban mask from {urban_mask.crs} to {src.crs}")
                urban_mask = urban_mask.to_crs(src.crs)
            mask = np.zeros((height, width), dtype=bool)
            for geom in urban_mask.geometry:
                if geom.geom_type in ['Polygon', 'MultiPolygon']:
                    rasterized = features.rasterize([(geom, 1)], out_shape=(height, width),
                                                    transform=src.transform, fill=0, dtype=np.uint8)
                    mask |= (rasterized == 1)
            urban_px = np.sum(mask)
            print(f"    Urban mask covers: {urban_px:,} pixels")
            # Stats before overwrite
            cls_in_urban, cnts_in_urban = np.unique(classification[mask], return_counts=True)
            print("    Classes in urban mask area BEFORE applying:")
            for cid, cnt in zip(cls_in_urban, cnts_in_urban):
                print(f"      {CLASSES.get(cid, {}).get('name', f'Class_{cid}')}: {cnt:,} px")
            classification[mask] = 3
            bare_after = np.sum(classification == 3)
            print(f"    Bare and Urban AFTER urban mask: {bare_after:,} pixels")
            print(f"    ✓ Urban mask applied to {urban_px:,} pixels")
        else:
            print("  - No urban mask available for this fire")
        return classification, src.transform, src.crs, band_info

def create_classification_visualization(classification):
    """
    (EN) Create RGB image colored by class.
    (BG) Създава RGB изображение, оцветено по класове.
    """
    vis = np.zeros((*classification.shape, 3), dtype=np.uint8)
    for cls_id, info in CLASSES.items():
        vis[classification == cls_id] = info['color']
    return vis

def export_classification_results(classification, transform, crs, fire_id, output_dir):
    """
    (EN) Export as GeoTIFF and vector GeoJSON.
    (BG) Експортира като GeoTIFF и векторен GeoJSON.
    """
    print("  - Exporting classification results...")
    base = f"{fire_id}_svc_classification"
    # GeoTIFF
    tif_path = os.path.join(output_dir, f"{base}.tif")
    with rasterio.open(tif_path, 'w', driver='GTiff', height=classification.shape[0],
                       width=classification.shape[1], count=1, dtype=classification.dtype,
                       crs=crs, transform=transform) as dst:
        dst.write(classification, 1)
        cmap = {cid: info['color'] for cid, info in CLASSES.items()}
        dst.write_colormap(1, cmap)
    print(f"    ✅ Raster saved: {tif_path}")

    # Vector
    vec_dir = os.path.join(output_dir, "vector_exports")
    os.makedirs(vec_dir, exist_ok=True)
    geojson_path = os.path.join(vec_dir, f"{base}.geojson")
    print("  - Polygonizing classification...")
    shapes_iter = features.shapes(classification.astype(np.int16), transform=transform)
    geoms, props = [], []
    for geom_dict, value in shapes_iter:
        if value > 0:
            geom = shape(geom_dict)
            if not geom.is_empty and geom.area > 100:
                geoms.append(geom)
                area_sqm = int(round(geom.area * (abs(transform[0]) ** 2)))
                props.append({
                    'class_id': int(value),
                    'class_name': CLASSES[int(value)]['name'],
                    'area_sqm': area_sqm,
                    'area_ha': area_sqm / 10000.0
                })
    if geoms:
        gdf = gpd.GeoDataFrame(props, geometry=geoms, crs=crs)
        gdf.to_file(geojson_path, driver='GeoJSON')
        print(f"    ✅ Vector export saved: {geojson_path} ({len(geoms)} polygons)")
    else:
        print("    ⚠ No valid polygons created")
    return tif_path, geojson_path

def create_classification_report(classification, band_info, fire_id, output_dir):
    """
    (EN) Generate combined report image (RGB, class map, pie, bar, table).
    (BG) Генерира обобщен доклад (RGB, карта, пай, стълбчета, таблица).
    """
    print("  - Generating classification report...")
    total_pixels = classification.size
    stats = {}
    for cls_id in np.unique(classification):
        if cls_id > 0:
            cnt = np.sum(classification == cls_id)
            stats[cls_id] = {
                'name': CLASSES[cls_id]['name'],
                'pixels': cnt,
                'percentage': (cnt / total_pixels) * 100,
                'color': CLASSES[cls_id]['color']
            }

    rgb_img = create_true_color_rgb(band_info)
    class_vis = create_classification_visualization(classification)

    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    fig.suptitle(f'SVC Classification - {fire_id}', fontsize=20, fontweight='bold', y=0.98)

    # RGB
    ax1 = axes[0, 0]
    if rgb_img is not None:
        ax1.imshow(rgb_img)
        ax1.set_title('TRUE COLOR RGB (Input Image)', fontsize=14, fontweight='bold')
    else:
        ax1.text(0.5, 0.5, 'RGB not available', ha='center')
    ax1.axis('off')

    # Classification map
    ax2 = axes[0, 1]
    ax2.imshow(class_vis)
    ax2.set_title('SVC Classification', fontsize=14, fontweight='bold')
    ax2.axis('off')

    # Pie chart
    ax3 = axes[1, 0]
    if stats:
        labels = [stats[c]['name'] for c in sorted(stats)]
        sizes = [stats[c]['percentage'] for c in sorted(stats)]
        colors = [tuple(comp/255 for comp in stats[c]['color']) for c in sorted(stats)]
        wedges, texts, autotexts = ax3.pie(sizes, labels=labels, colors=colors,
                                           autopct='%1.1f%%', startangle=90)
        for at in autotexts:
            at.set_color('white')
            at.set_fontweight('bold')
        ax3.set_title('Land Cover Distribution', fontsize=14, fontweight='bold')
        ax3.axis('equal')

    # Bar chart
    ax4 = axes[1, 1]
    if stats:
        ids = sorted(stats)
        names = [stats[c]['name'] for c in ids]
        pcts = [stats[c]['percentage'] for c in ids]
        colors = [tuple(c/255 for c in stats[c]['color']) for c in ids]
        bars = ax4.bar(range(len(names)), pcts, color=colors, edgecolor='black')
        ax4.set_xticks(range(len(names)))
        ax4.set_xticklabels(names, rotation=45, ha='right')
        ax4.set_ylabel('Percentage (%)')
        ax4.grid(True, alpha=0.3)
        for bar, pct in zip(bars, pcts):
            ax4.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                     f'{pct:.1f}%', ha='center', fontsize=9, fontweight='bold')

    # Table
    ax5 = axes[2, 0]
    ax5.axis('off')
    if stats:
        table_data = []
        total_area = 0
        total_px = 0
        for c in sorted(stats):
            s = stats[c]
            area_ha = s['pixels'] * 100 / 10000   # assuming 10 m pixel
            total_area += area_ha
            total_px += s['pixels']
            table_data.append([s['name'], f"{s['pixels']:,}", f"{s['percentage']:.2f}%", f"{area_ha:.1f}"])
        table_data.append(['TOTAL', f"{total_px:,}", '100.00%', f"{total_area:.1f}"])
        tbl = ax5.table(cellText=table_data, colLabels=['Land Cover','Pixels','%','Area (ha)'],
                        cellLoc='center', loc='center', colWidths=[0.4, 0.2, 0.2, 0.2])
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(9)
        tbl.scale(1.2, 1.8)
        for (row, col), cell in tbl.get_celld().items():
            if row == 0:
                cell.set_text_props(fontweight='bold', color='white')
                cell.set_facecolor('#2E86C1')
            elif row == len(table_data):
                cell.set_text_props(fontweight='bold')
                cell.set_facecolor('#F0F0F0')

    # Summary text
    ax6 = axes[2, 1]
    ax6.axis('off')
    if stats:
        legend_elems = [mpatches.Patch(color=tuple(c/255 for c in stats[cl]['color']),
                                       label=stats[cl]['name']) for cl in sorted(stats)]
        ax6.legend(handles=legend_elems, loc='upper center', fontsize=10, ncol=2)
        domin = max(stats.items(), key=lambda x: x[1]['percentage'])
        forest_pct = sum(stats[c]['percentage'] for c in [5,6] if c in stats)
        veg_pct = sum(stats[c]['percentage'] for c in [4,5,6] if c in stats)
        bare_pct = sum(stats[c]['percentage'] for c in [3] if c in stats)
        summary = [
            f"CLASSIFICATION SUMMARY (SVC):",
            f"Total area: {total_area:.1f} ha",
            f"Dominant: {domin[1]['name']} ({domin[1]['percentage']:.1f}%)",
            f"Forest: {forest_pct:.1f}%",
            f"Vegetation: {veg_pct:.1f}%",
            f"Bare/Urban: {bare_pct:.1f}%",
            f"Water: {stats.get(2,{}).get('percentage',0):.1f}%"
        ]
        ax6.text(0.02, 0.6, "\n".join(summary), transform=ax6.transAxes, fontsize=10,
                 verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.7))

    plt.tight_layout()
    report_path = os.path.join(output_dir, f"{fire_id}_classification_report.png")
    plt.savefig(report_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"    ✅ Report saved: {report_path}")

    # CSV statistics
    csv_path = os.path.join(output_dir, f"{fire_id}_classification_statistics.csv")
    stat_rows = [{'class_id': c, 'class_name': stats[c]['name'], 'pixels': stats[c]['pixels'],
                  'percentage': stats[c]['percentage'], 'area_ha': stats[c]['pixels']*100/10000}
                 for c in sorted(stats)]
    pd.DataFrame(stat_rows).to_csv(csv_path, index=False)
    print(f"    ✅ Statistics saved: {csv_path}")
    return report_path, csv_path

def display_classification_report(report_path, fire_id):
    """
    (EN) Display the report PNG if the file exists.
    (BG) Показва PNG доклада, ако файлът съществува.
    """
    print(f"\n📊 DISPLAYING REPORT FOR {fire_id}")
    if os.path.exists(report_path):
        display(HTML(f"<h2 style='color: #2E86C1;'>📊 SVC Classification Report - {fire_id}</h2>"))
        display(Image(filename=report_path, width=1200))
        print(f"📁 Report path: {report_path}")
    else:
        print(f"⚠ Report file not found: {report_path}")

# ==================== MAIN PROCESS / ОСНОВЕН ПРОЦЕС ====================
def process_fire_by_number(fire_num):
    """
    (EN) Full pipeline for one fire. Skips fire if no training points or classifier fails.
    (BG) Цялостен pipeline за един пожар. Пропуска при липса на точки или неуспех на класификатора.
    """
    print(f"\n{'='*60}")
    print(f"🔥 PROCESSING FIRE {fire_num:02d}")
    print(f"{'='*60}")
    fire_id_dict = get_fire_id_formats(fire_num)
    display_id = fire_id_dict['fire_01']

    # Find image file / Намиране на изображението
    image_file = None
    patterns = [
        f"fire{fire_num}_simple_average_all_bands_2024.tif",
        f"fire_{fire_num}_simple_average_all_bands_2024.tif",
        f"fire{fire_num}_simple_average.tif",
        f"fire_{fire_num}_simple_average.tif"
    ]
    for pat in patterns:
        fpath = os.path.join(PREVIEW_DIR, pat)
        if os.path.exists(fpath):
            image_file = pat
            break
    if not image_file:
        print(f"❌ No image file for fire {fire_num}")
        return None
    img_path = os.path.join(PREVIEW_DIR, image_file)

    try:
        # Load polygons / Зареждане на полигони
        print("📚 Loading training polygons...")
        train_pts, train_lbls, poly_crs = load_polygon_samples(fire_id_dict)
        if train_pts is None or len(train_pts) == 0:
            print(f"❌ No training samples for fire {fire_num}")
            return None
        print(f"  ✓ Total training samples: {len(train_pts)}")

        # Load urban mask / Зареждане на урбанизирана маска
        print("🏙️  Loading urban mask...")
        urban_mask = load_urban_mask(fire_id_dict)

        # Train SVC classifier / Обучение на SVC класификатор
        print("🎯 Training SVC classifier...")
        clf, scaler, pca = train_svc_classifier(img_path, train_pts, train_lbls, poly_crs)
        if clf is None:
            print("❌ Failed to train classifier (no valid training pixels). Skipping fire.")
            return None

        # Classify raster / Класифициране на растер
        print("🖼️  Classifying raster...")
        classif, transform, raster_crs, band_info = classify_raster_with_svc(img_path, clf, scaler, pca, urban_mask)

        # CRS for export – use CSV CRS if available, else raster CRS
        csv_crs = get_crs_for_fire(fire_num)
        export_crs = csv_crs if csv_crs else raster_crs
        if csv_crs and str(csv_crs) != str(raster_crs):
            print(f"  ⚠ Reprojecting from {raster_crs} to {csv_crs}...")
            classif, transform = reproject_classification(classif, transform, raster_crs, csv_crs)
            print("  ✅ Reprojected")

        # Export / Експорт
        print("💾 Exporting results...")
        rast_path, vec_path = export_classification_results(classif, transform, export_crs, display_id, CLASSIFIED_DIR)
        report_path, stats_path = create_classification_report(classif, band_info, display_id, CLASSIFIED_DIR)

        # Save classifier / Запазване на модела
        clf_path = os.path.join(CLASSIFIED_DIR, f"{display_id}_svc_classifier.joblib")
        joblib.dump({'classifier': clf, 'scaler': scaler, 'pca': pca}, clf_path)
        print(f"    ✅ Classifier saved: {clf_path}")

        # Display report / Показване на доклад
        display_classification_report(report_path, display_id)

        return {
            'fire_id': display_id,
            'fire_num': fire_num,
            'filename': image_file,
            'classifier': clf,
            'scaler': scaler,
            'pca': pca,
            'raster_path': rast_path,
            'vector_path': vec_path,
            'report_path': report_path,
            'stats_path': stats_path,
            'classifier_path': clf_path,
            'training_samples': len(train_pts),
            'urban_mask_used': urban_mask is not None,
            'export_crs': str(export_crs)
        }
    except Exception as e:
        print(f"❌ Error processing fire {fire_num}: {e}")
        import traceback
        traceback.print_exc()
        return None

def main():
    """
    (EN) Main entry: discover fires, process sequentially, generate summary.
    (BG) Основна точка: открива пожарите, обработва ги последователно и прави обобщение.
    """
    print("🚀 Starting SVC Land Cover Classification")
    print(f"📁 Source directory: {PREVIEW_DIR}")
    print(f"📊 Output directory: {CLASSIFIED_DIR}")
    print(f"📚 Polygon directory: {POLYGON_DIR}")
    print(f"🏙️  Urban mask directory: {URBAN_MASK_DIR}")
    print("="*80)

    if not os.path.exists(PREVIEW_DIR):
        print(f"❌ Source directory not found: {PREVIEW_DIR}")
        sys.exit(1)
    if not os.path.exists(POLYGON_DIR):
        print(f"❌ Polygon directory not found: {POLYGON_DIR}")
        sys.exit(1)

    urban_mask_exists = os.path.exists(URBAN_MASK_DIR)
    if urban_mask_exists:
        urban_files = [f for f in os.listdir(URBAN_MASK_DIR) if f.endswith('.gpkg')]
        print(f"✓ Urban mask directory found, {len(urban_files)} files")
    else:
        print("⚠ Urban mask directory not found, continuing without masks")

    # Find max fire number / Намиране на максимален номер на пожар
    fire_nums = []
    for f in os.listdir(PREVIEW_DIR):
        if f.endswith(('.tif', '.tiff')):
            num = extract_fire_number_from_filename(f)
            if num:
                fire_nums.append(num)
    if not fire_nums:
        print("❌ No fire image files found")
        sys.exit(1)
    max_fire = max(fire_nums)
    print(f"📊 Found fire images up to fire {max_fire}")

    results = []
    processed = 0
    for fire_num in range(1, max_fire+1):
        print(f"\n{'='*80}")
        print(f"PROCESSING FIRE {fire_num:02d} / {max_fire}")
        print(f"{'='*80}")
        res = process_fire_by_number(fire_num)
        if res:
            results.append(res)
            processed += 1
            print(f"\n✅ Successfully processed fire {fire_num:02d}")
        else:
            print(f"\n⚠ Skipping fire {fire_num:02d} (insufficient data or error)")

    # Summary / Обобщение
    if results:
        print(f"\n{'='*80}")
        print("📊 SVC CLASSIFICATION SUMMARY")
        summary_data = []
        for r in results:
            if os.path.exists(r['stats_path']):
                sdf = pd.read_csv(r['stats_path'])
                total_area = sdf['area_ha'].sum()
                summary_data.append({
                    'Fire ID': r['fire_id'],
                    'Fire #': r['fire_num'],
                    'Training Samples': r['training_samples'],
                    'Urban Mask Used': 'Yes' if r.get('urban_mask_used') else 'No',
                    'Export CRS': r.get('export_crs', 'N/A'),
                    'Classes': len(sdf),
                    'Total Area (ha)': f"{total_area:.1f}",
                    'Report': os.path.basename(r['report_path'])
                })
        if summary_data:
            summ_df = pd.DataFrame(summary_data)
            print("\n📋 Processing Summary:\n", summ_df.to_string(index=False))
            summ_path = os.path.join(CLASSIFIED_DIR, "svc_processing_summary.csv")
            summ_df.to_csv(summ_path, index=False)
            print(f"\n💾 Summary saved: {summ_path}")
            mask_count = sum(1 for r in results if r.get('urban_mask_used'))
            print(f"🏙️  Urban masks applied to {mask_count} out of {len(results)} fires")
        print(f"\n✅ Successfully processed {processed}/{max_fire} fires")
    else:
        print("\n⚠ No fires were processed successfully.")

    print(f"\n🎯 Processing complete! Results in {CLASSIFIED_DIR}")
    print("="*80)

if __name__ == "__main__":
    main()